Практика 1, 12 минут
[База] Сколько заказов и какая выручка оплаченных в каждой категории? Отсортируйте по выручке. Сходится с ДЗ 4 (лидер Электроника, 34,4 млн ₸)?
[База] Таблица по городам: число заказов и доля отмен (agg с именами). Какой город выделяется и на скольких заказах это видно?
[Стандарт] Средний и медианный чек оплаченных заказов по городам (agg(["mean", "median"])). Может, Шымкент падает, потому что там покупают дешевле?
[Стандарт] Доля отмен по сроку доставки: groupby("delivery_days"), рядом число заказов. Как отмены зависят от срока?
[Челлендж] Средний срок доставки по городам и месяцам (groupby по двум столбцам + unstack). Какая клетка выделяется? Почему count и size в этой таблице дают разные числа?

In [19]:
# [База] Сколько заказов и какая выручка оплаченных в каждой категории? 
# Отсортируйте по выручке. Сходится с ДЗ 4 (лидер Электроника, 34,4 млн ₸)?

import pandas as pd
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

df = pd.read_csv("orders.csv")
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
df["city"] = df["city"].str.strip().str.lower().map(CITY)
df = df.drop_duplicates()
df["order_date"] = pd.to_datetime(df["order_date"])
df["month"] = df["order_date"].dt.month
df = df[df["customer_id"] != "TEST"].copy()    # copy: дальше добавим столбцы
df.shape

(5280, 9)

In [20]:
df["cancel"] = df["status"] == "отменён"     # True/False: пригодится для доли отмен
paid = df[df["status"] == "оплачен"]
len(paid)

4739

In [21]:
paid.groupby("category").agg(
    order_count=('order_id', "count"),
    rev=('amount', "sum"))

,order_count,rev
category,,
Дом,952,10123520.0
Книги,564,2206210.0
Красота,857,5289940.0
Одежда,1330,17672830.0
Электроника,1036,34370430.0


In [22]:
# [База] Таблица по городам: число заказов и доля отмен (agg с именами). 
# Какой город выделяется и на скольких заказах это видно?

by_city = df.groupby("city").agg(orders=("order_id", "count"),
                                 cancel_share=("cancel", "mean"),
                                 avg_days=("delivery_days", "mean"))
by_city.sort_values("cancel_share").round(4)

,orders,cancel_share,avg_days
city,,,
Атырау,143,0.0769,2.0362
Алматы,2116,0.0775,1.9971
Астана,1339,0.0881,2.0192
Караганда,519,0.0906,1.9414
Актобе,387,0.1059,2.0705
Шымкент,776,0.2062,4.4509


In [28]:
# [Стандарт] Средний и медианный чек оплаченных заказов по городам (agg(["mean", "median"])). 
# Может, Шымкент падает, потому что там покупают дешевле?

paid.groupby("city")["amount"].agg(["mean", "median"])

,mean,median
city,,
Актобе,14790.115607,10210.0
Алматы,14930.373975,10445.0
Астана,14647.624898,10120.0
Атырау,14347.348485,9920.0
Караганда,14873.305085,10455.0
Шымкент,13965.357143,9995.0


In [30]:
# [Стандарт] Доля отмен по сроку доставки: groupby("delivery_days"), рядом число заказов. 
# Как отмены зависят от срока?

df.groupby("delivery_days")["cancel"].mean()

delivery_days
1.0    0.042389
2.0    0.079618
3.0    0.133962
5.0    0.220930
6.0    0.216667
7.0    0.347368
8.0    0.432692
Name: cancel, dtype: float64

In [34]:
# [Челлендж] Средний срок доставки по городам и месяцам (groupby по двум столбцам + unstack). 
# Какая клетка выделяется?
#  Почему count и size в этой таблице дают разные числа?

df.groupby(["city", "month"])["delivery_days"].agg(["mean","count", "size"]).unstack()

mean           count        size      
month            9         10    9     10    9     10
city                                                 
Актобе     2.042553  2.097436   188   195   190   197
Алматы     1.981132  2.012428  1007  1046  1043  1073
Астана     2.045031  1.993893   644   655   665   674
Атырау     2.060606  2.013889    66    72    69    74
Караганда  1.896552  1.991453   261   234   274   245
Шымкент    2.031519  6.535802   349   405   359   417